In [1]:
import torch
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import numpy as np
import cv2
import os
from torch import nn

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import matplotlib.pyplot as plt



In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')




def preprocess_image(image, train=True):

    img = np.array(image)

    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    l, a, b = cv2.split(img)

    clahe = cv2.createCLAHE(clipLimit=2.0)
    cl = clahe.apply(l)

    limg = cv2.merge((cl, a, b))
    img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    image = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if train:

        transform = transforms.Compose([
            transforms.Resize((256,256)),
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225]
            )
        ])

    else:

        transform = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225]
            )
        ])

    image_tensor = transform(image)

    return image_tensor



In [3]:
def collect_images(dataset_path):

    classes = sorted(os.listdir(dataset_path))
    class_to_idx = {c:i for i,c in enumerate(classes)}

    images = []

    valid_ext = (".jpg", ".jpeg", ".png")

    for c in classes:

        class_folder = os.path.join(dataset_path, c)

        for img in os.listdir(class_folder):

            if not img.lower().endswith(valid_ext):
                continue

            path = os.path.join(class_folder, img)

            images.append((path, class_to_idx[c]))

    return images, classes,class_to_idx

def extract_leaf_mask(image):
    hsv_image=cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    LOWER_LEAF = np.array([15, 40, 40])
    UPPER_LEAF = np.array([95, 255, 255])

    KERNEL = np.ones((5, 5), np.uint8)

    mask = cv2.inRange(hsv_image, LOWER_LEAF, UPPER_LEAF)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, KERNEL)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None, 0

    largest = max(contours, key=cv2.contourArea)

    final_leaf_mask = np.zeros_like(mask)
    cv2.drawContours(final_leaf_mask, [largest], -1, 255, -1)

    leaf_area = cv2.countNonZero(final_leaf_mask)
    leaf = cv2.bitwise_and(image, image, mask=final_leaf_mask)

    return leaf, final_leaf_mask

In [4]:
class LeafDataset(Dataset):

    def __init__(self, data, train=True):

        self.data = data
        self.train = train

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        img_path, label = self.data[idx]

        with Image.open(img_path) as img:
            image = img.convert("RGB")

        image = preprocess_image(image, train=self.train)

        return image, label

In [5]:
dataset_path="Datasets/data_set"

images, classes, class_to_idx = collect_images(dataset_path)

train_data, temp_data = train_test_split(
    images,
    test_size=0.3,
    stratify=[label for _,label in images],
    random_state=42
)
labels = [label for _, label in train_data]

val_data, test_data = train_test_split(
    temp_data,
    test_size=0.5,
    stratify=[label for _,label in temp_data],
    random_state=42
)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

In [6]:
train_dataset = LeafDataset(train_data, train=True)
val_dataset   = LeafDataset(val_data, train=False)
test_dataset  = LeafDataset(test_data, train=False)

In [7]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [8]:
class_num=len(classes)
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
model.classifier = nn.Linear(model.classifier.in_features, class_num)
model=model.to(device)
loss_fun=nn.CrossEntropyLoss(weight=class_weights)
epochs=20
patience = 5
best_val_loss = float("inf")
counter = 0


In [9]:

freeze_epochs = 3

for epoch in range(epochs):

    if epoch == 0:
        # Freeze backbone
        for param in model.parameters():
            param.requires_grad = False

        for param in model.classifier.parameters():
            param.requires_grad = True

        optimizer = torch.optim.Adam(model.classifier.parameters(), lr=3e-4)

    if epoch == freeze_epochs:
        print("Unfreezing DenseNet...")

        for param in model.parameters():
            param.requires_grad = True

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = loss_fun(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        loop.set_postfix(
            loss=running_loss/len(train_loader),
            acc=100*correct/total
        )

    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = loss_fun(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct / total

    print(f"\nValidation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        counter = 0

        torch.save(model.state_dict(), "best_model.pth")

        print("Validation improved, model saved!")

    else:

        counter += 1
        print(f"No improvement for {counter} epochs")

        if counter >= patience:
            print("Early stopping triggered!")
            break

Epoch 1/20: 100%|██████████| 452/452 [01:15<00:00,  5.96it/s, acc=55.4, loss=1.62] 



Validation Loss: 1.0299
Validation Accuracy: 75.97%
Validation improved, model saved!


Epoch 2/20: 100%|██████████| 452/452 [01:06<00:00,  6.85it/s, acc=72.9, loss=1.05] 



Validation Loss: 0.7457
Validation Accuracy: 79.81%
Validation improved, model saved!


Epoch 3/20: 100%|██████████| 452/452 [01:06<00:00,  6.81it/s, acc=76.8, loss=0.861]


...


Validation Loss: 0.0374
Validation Accuracy: 98.71%
No improvement for 2 epochs


Epoch 19/20: 100%|██████████| 452/452 [02:05<00:00,  3.60it/s, acc=98, loss=0.0599]  



Validation Loss: 0.0296
Validation Accuracy: 99.32%
No improvement for 3 epochs


Epoch 20/20: 100%|██████████| 452/452 [02:05<00:00,  3.60it/s, acc=97.9, loss=0.0617]



Validation Loss: 0.0402
Validation Accuracy: 98.90%
No improvement for 4 epochs


In [11]:
#files.upload()
model.load_state_dict(torch.load("best_model.pth",map_location=device,weights_only=True))

<All keys matched successfully>

In [12]:
from sklearn.metrics import classification_report
model.eval()

test_loss = 0
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    loop = tqdm(test_loader, desc="Testing")

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = loss_fun(outputs, labels)

        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_loss /= len(test_loader)
test_acc = 100 * correct / total

print("\nTest Results")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=classes))

Testing: 100%|██████████| 97/97 [00:14<00:00,  6.58it/s]


Test Results
Test Loss: 0.0181
Test Accuracy: 99.29%

Classification Report:
                        precision    recall  f1-score   support

        Bacterial_spot       0.99      0.99      0.99       468
          Early_blight       0.99      0.98      0.98       300
             Leaf_Mold       1.00      1.00      1.00       143
    Septoria_leaf_spot       1.00      1.00      1.00       266
           Target_Spot       1.00      0.99      0.99       210
YellowLeaf__Curl_Virus       1.00      0.99      0.99       482
               healthy       1.00      0.99      1.00       483
           late_blight       0.98      1.00      0.99       437
          mosaic_virus       1.00      1.00      1.00        56
           spider_mite       0.99      1.00      0.99       251

              accuracy                           0.99      3096
             macro avg       0.99      0.99      0.99      3096
          weighted avg       0.99      0.99      0.99      3096



In [13]:
# target_layers = [model.features.denseblock4.denselayer16.conv2]
target_layers = [model.features[-2]]
threshold = 90
output_dir = "gradcam_results"
os.makedirs(output_dir, exist_ok=True)
cam = GradCAMPlusPlus(model=model, target_layers=target_layers)

labels_dir = "yolo_labels"
os.makedirs(labels_dir, exist_ok=True)


In [14]:
INPUT_BASE = "Datasets/data_set"
OUTPUT_BASE = "Datasets/data_set_generateddense"

gradcam_dirs = [
    INPUT_BASE  # since you didn't split before
]

unet_mask_dir = "unet_masksdense"
output_dir = "gradcam_resultsdense"

os.makedirs(unet_mask_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

cam = GradCAMPlusPlus(
    model=model,
    target_layers=target_layers
)

counter = 0

for dataset_dir in gradcam_dirs:

    yolo_labels_dir = os.path.join(OUTPUT_BASE, "labels")
    yolo_images_dir = os.path.join(OUTPUT_BASE, "images")

    os.makedirs(yolo_labels_dir, exist_ok=True)
    os.makedirs(yolo_images_dir, exist_ok=True)

    for class_name in os.listdir(dataset_dir):

        class_folder = os.path.join(dataset_dir, class_name)

        if not os.path.isdir(class_folder):
            continue

        for img_name in tqdm(os.listdir(class_folder), desc=class_name):

            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            counter += 1

            img_path = os.path.join(class_folder, img_name)
            base_name = os.path.splitext(img_name)[0]

            yolo_img_path = os.path.join(yolo_images_dir, img_name)
            label_path = os.path.join(yolo_labels_dir, f"{base_name}.txt")

            try:
                # LOAD IMAGE
                image = Image.open(img_path).convert("RGB")
                original_img = cv2.resize(np.array(image), (224, 224))

                # LEAF MASK
                leaf, leaf_mask = extract_leaf_mask(original_img)
                if leaf is None:
                    continue

                rgb_img = original_img.astype(np.float32) / 255.0

                image_tensor = preprocess_image(
                    Image.fromarray(original_img),
                    train=False
                ).unsqueeze(0).to(device)

                # PREDICTION
                outputs = model(image_tensor)
                class_id = outputs.argmax(dim=1).item()
                targets = [ClassifierOutputTarget(class_id)]

                is_healthy = "healthy" in class_name.lower()

                # SAVE IMAGE
                if not os.path.exists(yolo_img_path):
                    cv2.imwrite(yolo_img_path, cv2.cvtColor(original_img, cv2.COLOR_RGB2BGR))

                # MASK LOGIC
                if is_healthy:
                    mask = leaf_mask
                    visualization = rgb_img

                else:
                    grayscale_cam = cam(input_tensor=image_tensor, targets=targets)[0]

                    leaf_mask_float = leaf_mask.astype(np.float32) / 255.0
                    grayscale_cam *= leaf_mask_float

                    cam_min, cam_max = grayscale_cam.min(), grayscale_cam.max()
                    if cam_max - cam_min == 0:
                        continue

                    heatmap_norm = (grayscale_cam - cam_min) / (cam_max - cam_min)

                    threshold = np.percentile(heatmap_norm, 85)
                    mask = (heatmap_norm >= threshold).astype("uint8") * 255

                    kernel = np.ones((3, 3), np.uint8)
                    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
                    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

                    visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

                # YOLO LABELS
                if not os.path.exists(label_path):

                    img_h, img_w = mask.shape
                    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

                    with open(label_path, "w") as f:
                        for contour in contours:

                            area = cv2.contourArea(contour)
                            if area < 0.001 * img_h * img_w:
                                continue

                            x, y, w, h = cv2.boundingRect(contour)
                            if w < 5 or h < 5:
                                continue

                            x_center = (x + w / 2) / img_w
                            y_center = (y + h / 2) / img_h
                            width = w / img_w
                            height = h / img_h

                            f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

                # SAVE UNET MASK
                mask_path = os.path.join(unet_mask_dir, f"{class_name}_{base_name}.png")
                cv2.imwrite(mask_path, mask)

                # VISUALIZATION
                mask_color = original_img.copy()
                mask_color[mask == 255] = [255, 0, 0]

                if visualization.dtype != np.uint8:
                    visualization = (visualization * 255).astype(np.uint8)

                combined = np.concatenate(
                    [original_img, visualization, mask_color],
                    axis=1
                )

                save_path = os.path.join(output_dir, f"{class_name}_{base_name}.jpg")
                cv2.imwrite(save_path, cv2.cvtColor(combined, cv2.COLOR_RGB2BGR))

            except Exception as e:
                print("Error:", img_name, e)

print(f"✅ DONE: {counter} images processed")

YellowLeaf__Curl_Virus: 100%|██████████| 3209/3209 [03:02<00:00, 17.54it/s]

✅ DONE: 20638 images processed
